## Visualize dataset for Omero

In [ ]:
import os
import numpy as np
import glob
import tifffile as tiff
import matplotlib.pyplot as plt


def normalize(img):
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, (1, 99))
    return np.clip((img - p1) / (p99 - p1 + 1e-6), 0, 1)

def overlay_images(dapi_img, hne_img):
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)

BASE_DIR = "images_2751_2790"

image_folders = sorted(glob.glob(os.path.join(BASE_DIR, "Image_27*")))

for folder in image_folders:
    print(f"\n[VIEW] {folder}")

    try:
        multiplex_path = glob.glob(os.path.join(folder, "rack*.ome.tif"))[0]
        hne_registered_path = [
            f for f in glob.glob(os.path.join(folder, "*.ome.tif"))
            if not os.path.basename(f).startswith("rack")
        ][0]

        id = folder.split("Image_")[1]

        # load images 
        hne_img = np.array(tiff.imread(hne_registered_path))[0,:,:]
        multiplex_img = np.array(tiff.imread(multiplex_path))[0,:,:]

        hne_ch0 = hne_img
        multiplex_ch0 = multiplex_img
        
        hne_n = 1 - normalize(hne_img)
        multiplex_n = normalize(multiplex_img)

        # create overlay 
        overlay = np.zeros((*hne_n.shape, 3))
        overlay[..., 0] = hne_n        # red = H&E
        overlay[..., 1] = multiplex_n  # green = multiplex

        # plot
        plt.subplot(1, 3, 1)
        plt.imshow(hne_n, cmap="gray")
        plt.title("Registered H&E")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(multiplex_n, cmap="gray")
        plt.title("Multiplexed")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        overlay_images(multiplex_n, hne_n)
        plt.title("Overlay")
        plt.axis("off")
    
        plt.suptitle(id, fontsize=14, y=1.02)

        plt.tight_layout()
        plt.show()


    except:
        print(f"[ERROR] Could not load images in {folder}")
        continue


    # ----- cannot register for following images due to not enough feature points -----
    # CRC017-rack-01-well-D01-roi-003 (2774)
    # CRC057-rack-01-well-C01-roi-002 (2783)
    # ---------------------------------------------------------------------------------